# 01 · Pipeline walkthrough

Runs every layer of `coop_assistant` on this machine, in the order a caller's question flows through it:

1. **Evidence store**: dated SQLite price records
2. **Decision engine**: deterministic `ANSWER / CLARIFY / ABSTAIN / REFER`
3. **Approved replies** in five languages
4. **Understanding**: rules first, a small local LLM only for gaps, plus the number guard
5. **Business Q&A** grounded in `data/business.json`
6. **A full conversation** through `Assistant`

> All prices and the business profile are **synthetic demo data**.

In [1]:
import json, os, pathlib, time, warnings
import pandas as pd
warnings.filterwarnings("ignore")
if pathlib.Path.cwd().name == "notebooks":
    os.chdir("..")
os.environ.pop("ELEVENLABS_API_KEY", None)  # local Piper voice only: these notebooks demonstrate the offline core
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

from coop_assistant import __version__, config
print("coop_assistant", __version__)
def shown(value):  # keep machine-specific paths out of the saved notebook
    return str(value).replace(str(config.PROJECT_ROOT), "<repo>").replace(str(pathlib.Path.home()), "~")

pd.Series({k: shown(getattr(config, k)) for k in
           ["MODEL_DIR", "BUSINESS_FILE", "OLLAMA_URL", "EXTRACT_MODEL", "QA_MODEL", "WHISPER_SIZE"]}, name="setting").to_frame()

coop_assistant 0.1.0


,setting
MODEL_DIR,~\AppData\Local\coffee-price-ai
BUSINESS_FILE,<repo>\data\business.json
OLLAMA_URL,http://localhost:11434
EXTRACT_MODEL,coop-extract
QA_MODEL,gemma3:4b
WHISPER_SIZE,small


## 1. Evidence store

Every price answer must point at one of these rows. The store is re-seeded on start, so the demo dates stay relative to today. That gives one valid record, one expired record and one with too few samples.

In [2]:
from coop_assistant.core.store import connect

conn = connect(":memory:")
pd.read_sql("""SELECT id, district, product_form, grade, price_value AS kes_per_kg, sample_count,
                      observed_at, valid_until, source_name FROM prices""", conn)

,id,district,product_form,grade,kes_per_kg,sample_count,observed_at,valid_until,source_name
0,DEMO-NYR-PARCH-A,Nyeri,parchment,A,120.0,12,2026-10-02,2026-10-09,SYNTHETIC DEMO FIXTURE
1,DEMO-NYR-CHERRY,Nyeri,cherry,ungraded,65.0,9,2026-10-02,2026-10-09,SYNTHETIC DEMO FIXTURE
2,DEMO-KMB-PARCH-A-STALE,Kiambu,parchment,A,110.0,8,2026-08-20,2026-09-04,SYNTHETIC DEMO FIXTURE
3,DEMO-KRN-PARCH-A-SPARSE,Kirinyaga,parchment,A,125.0,1,2026-10-03,2026-10-10,SYNTHETIC DEMO FIXTURE


## 2. Deterministic decision engine

No model is involved here. The engine either finds one matching, in-date, well-sampled record and returns its evidence ID, or it refuses to give a number.

In [3]:
from coop_assistant.core import decision as engine

def slots(**kw):
    s = engine.empty_slots()
    s.update(kw)
    return s

cases = {
    "valid match + offer":   ("PRICE_CHECK", slots(quote=105, product_form="parchment", grade="A", district="Nyeri", unit="kg")),
    "cherry, no grade needed": ("PRICE_CHECK", slots(product_form="cherry", district="Nyeri")),
    "stale record":          ("PRICE_CHECK", slots(product_form="parchment", grade="A", district="Kiambu")),
    "grade not in records":  ("PRICE_CHECK", slots(product_form="parchment", grade="B", district="Nyeri")),
    "too few samples":       ("PRICE_CHECK", slots(product_form="parchment", grade="A", district="Kirinyaga")),
    "no data for district":  ("PRICE_CHECK", slots(product_form="parchment", grade="A", district="Murang'a")),
    "grade missing":         ("PRICE_CHECK", slots(product_form="parchment", district="Nyeri")),
    "priced per bag":        ("PRICE_CHECK", slots(quote=5000, unit="bag", product_form="parchment", grade="A", district="Nyeri")),
    "foreign currency":      ("PRICE_CHECK", slots(currency="INR", product_form="cherry", district="Nyeri")),
    "asks for a person":     ("HUMAN", slots()),
}
keep = ("reference", "difference", "missing", "observed_at", "sample_count")
pd.DataFrame([
    {"case": name, "state": d["state"], "reason": d["reason"], "evidence_id": d["evidence_id"],
     "facts": {k: v for k, v in d["facts"].items() if k in keep}}
    for name, (intent, s) in cases.items()
    for d in [engine.decide(intent, s, conn)]
])

,case,state,reason,evidence_id,facts
0,valid match + offer,ANSWER,matched,DEMO-NYR-PARCH-A,"{'reference': 120.0, 'observed_at': '2026-10-02', 'sample_count': 12, 'difference': -15.0}"
1,"cherry, no grade needed",ANSWER,matched,DEMO-NYR-CHERRY,"{'reference': 65.0, 'observed_at': '2026-10-02', 'sample_count': 9}"
2,stale record,ABSTAIN,stale,NaN,"{'reference': 110.0, 'observed_at': '2026-08-20', 'sample_count': 8}"
3,grade not in records,ABSTAIN,no_grade,NaN,{}
4,too few samples,ABSTAIN,sparse,NaN,"{'reference': 125.0, 'observed_at': '2026-10-03', 'sample_count': 1}"
5,no data for district,ABSTAIN,no_data,NaN,{}
6,grade missing,CLARIFY,missing,NaN,{'missing': ['grade']}
7,priced per bag,CLARIFY,unit,NaN,{}
8,foreign currency,ABSTAIN,currency,NaN,{}
9,asks for a person,REFER,human_requested,NaN,{}


## 3. Approved replies in five languages

Templates are filled only from the record (120, the date, the market) and the caller's own words (105). The difference (15) is computed in code.

In [4]:
from coop_assistant.core.responses import SUPPORTED, render

answer = engine.decide(*cases["valid match + offer"], conn)
for lang in SUPPORTED:
    print(f"[{lang}] {render(answer, lang)}\n")
print("[sw, stale] " + render(engine.decide(*cases["stale record"], conn), "sw"))

[en] Just so you know, this is demo data. According to Demo Co-op Nyeri's records from October 2, parchment coffee grade A in Nyeri is going for 120 shillings a kilo. So the 105 you were offered is 15 shillings below that. It's your call whether to sell, and do confirm the grade and moisture with your co-op.

[hi] [डेमो डेटा] Nyeri में पार्चमेंट कॉफ़ी ग्रेड A का संदर्भ भाव 120 KES प्रति किलो है (Demo Co-op Nyeri, 2026-10-02 का, स्रोत: SYNTHETIC DEMO FIXTURE)। व्यापारी का 105 का भाव इससे 15 KES कम है। यह सिर्फ़ जानकारी है, बेचने का फ़ैसला आपका है। ग्रेड और नमी की पुष्टि अपनी सहकारी समिति से करें।

[sw] [DATA YA MAJARIBIO] Bei ya marejeleo ya kahawa ya parchment daraja A huko Nyeri ni KES 120 kwa kilo (Demo Co-op Nyeri, tarehe 2026-10-02, chanzo: SYNTHETIC DEMO FIXTURE). Bei ya mnunuzi ya 105 iko chini kwa KES 15 kuliko bei hii. Hii ni taarifa tu; uamuzi wa kuuza ni wako. Thibitisha daraja na unyevu na chama chako cha ushirika.

[zh] 【演示数据】Nyeri羊皮纸咖啡豆（A级）的参考价格为每公斤 120 KES（Demo Co-op Nyer

## 4. Understanding: rules first, small LLM for gaps

Rules handle digits, known words and district names instantly. When they leave required fields empty, `qwen2.5:3b` (local, via Ollama) extracts JSON under a strict schema. Either way the result goes through the same normalisation.

In [5]:
from coop_assistant.nlu import extract

sentences = [
    ("en", "Buyer offered 105 shillings per kg for grade A parchment coffee in Nyeri"),
    ("hi", "न्येरी में व्यापारी मेरी ग्रेड ए पार्चमेंट कॉफ़ी 105 शिलिंग किलो में ले रहा है, ठीक है क्या?"),
    ("sw", "Mnunuzi ametoa shilingi 105 kwa kilo kwa kahawa ya parchment daraja A huko Nyeri"),
    ("zh", "我在涅里，商人给我的A级羊皮纸咖啡每公斤105先令，合理吗？"),
    ("ko", "니에리에서 상인이 A등급 파치먼트 커피를 킬로당 105실링에 사겠대요"),
    ("en", "he is paying me 105 for my cherries here in nyeri"),
]
rows = []
for lang, sentence in sentences:
    start = time.time()
    intent, found, method = extract.extract(sentence)
    rows.append({"lang": lang, "method": method, "seconds": round(time.time() - start, 1), "intent": intent, **found})
pd.DataFrame(rows)

,lang,method,seconds,intent,quote,currency,unit,product_form,grade,district
0,en,regex,0.0,PRICE_CHECK,105.0,KES,kg,parchment,A,Nyeri
1,hi,llm,3.3,PRICE_CHECK,105.0,KES,kg,parchment,A,Nyeri
2,sw,regex,0.0,PRICE_CHECK,105.0,KES,kg,parchment,A,Nyeri
3,zh,llm,3.4,PRICE_CHECK,105.0,KES,kg,parchment,A,Nyeri
4,ko,llm,3.1,PRICE_CHECK,105.0,KES,kg,parchment,A,Nyeri
5,en,regex,0.0,PRICE_CHECK,105.0,NaN,NaN,cherry,NaN,Nyeri


### Guard: a price must actually be spoken

If a model returns a price whose digits are not in the transcript, the price is dropped. Spoken-word numbers ("mia moja na tano") are dropped too, so the system asks rather than guesses.

In [6]:
print(extract.normalize({"intent": "PRICE_CHECK", "quote": 999}, "the buyer offered 105 per kg"))
print(extract.normalize({"intent": "PRICE_CHECK", "quote": 1}, "shilingi mia moja na tano"))

('PRICE_CHECK', {'quote': None, 'currency': None, 'unit': None, 'product_form': None, 'grade': None, 'district': None})
('PRICE_CHECK', {'quote': None, 'currency': None, 'unit': None, 'product_form': None, 'grade': None, 'district': None})


## 5. Business Q&A grounded in `business.json`

`gemma3:4b` answers like a receptionist, but only from the profile. If an answer contains a number that is in neither the profile nor the question, it is blocked.

In [7]:
from coop_assistant.nlu import business_qa

for lang, question in [("en", "How do I become a member and what does it cost?"),
                       ("en", "When do you pay farmers?"),
                       ("hi", "मुझे एडवांस पैसे मिल सकते हैं क्या?"),
                       ("zh", "你们卖咖啡苗吗？多少钱？"),
                       ("en", "Do you sell tractors?")]:
    start = time.time()
    state, text = business_qa.answer(question, lang)
    print(f"[{lang}] {question}\n   -> {state} in {time.time() - start:.1f}s: {text}\n")

[en] How do I become a member and what does it cost?
   -> ANSWER in 13.8s: You can become a member by visiting the office during our opening hours. The one-time joining fee is KES 500 and you will need your National ID and proof of land use to register.



[en] When do you pay farmers?
   -> ANSWER in 12.4s: Members are paid monthly, on the 15th, by mobile money.



[hi] मुझे एडवांस पैसे मिल सकते हैं क्या?
   -> ANSWER in 14.4s: सदस्य अपने रिकॉर्ड किए गए डिलीवरी के विरुद्ध अधिकतम 30 प्रतिशत की अग्रिम राशि का अनुरोध कर सकते हैं। कृपया ध्यान दें कि हम कभी भी पासवर्ड या मोबाइल मनी पिन नहीं मांगते हैं।



[zh] 你们卖咖啡苗吗？多少钱？
   -> NO_INFO in 15.9s: None



[en] Do you sell tractors?
   -> NO_INFO in 14.3s: None



## 6. A full conversation through `Assistant`

The router sends price checks to the engine and everything else to Q&A. It keeps a pending clarification across turns and hands requests for a person straight to `REFER`.

In [8]:
from coop_assistant.pipeline import Assistant

bot = Assistant(connect(":memory:"))
for lang, text in [("en", "Buyer offered 105 for parchment coffee in Nyeri"),
                   ("en", "grade A"),
                   ("en", "Parchment grade A in Kiambu"),
                   ("en", "When do you pay farmers?"),
                   ("en", "Can I talk to somebody please?")]:
    r = bot.handle(text, lang)
    d = r["decision"]
    print(f"caller: {text}\n  {d['state']}/{d['reason']}  evidence={d['evidence_id']}\n  reply: {r['reply']}\n")

caller: Buyer offered 105 for parchment coffee in Nyeri
  CLARIFY/missing  evidence=None
  reply: Please tell me the grade.



caller: grade A
  ANSWER/matched  evidence=DEMO-NYR-PARCH-A
  reply: Just so you know, this is demo data. According to Demo Co-op Nyeri's records from October 2, parchment coffee grade A in Nyeri is going for 120 shillings a kilo. So the 105 you were offered is 15 shillings below that. It's your call whether to sell, and do confirm the grade and moisture with your co-op.

caller: Parchment grade A in Kiambu
  ABSTAIN/stale  evidence=None
  reply: Just so you know, this is demo data. My latest record for Kiambu is from 2026-08-20 and is out of date, so I cannot give a current price. Please check with your co-op.



caller: When do you pay farmers?
  ANSWER/business_info  evidence=BUSINESS-PROFILE
  reply: Members are paid monthly, on the 15th, by mobile money.

caller: Can I talk to somebody please?
  REFER/human_requested  evidence=None
  reply: With your permission, a co-op officer will call you back.

